## Cultural Places to Bronze

In [0]:
dbutils.widgets.text("file_name", "cultural_places.json")
FILE_NAME = dbutils.widgets.get("file_name")

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name
import json

date_utc = datetime.now(timezone.utc).strftime('%Y_%m_%d')

full_file_name = f"{FILE_NAME}_{date_utc}.json"

source_path = f"abfss://landing@stroutemindeuskadidev.dfs.core.windows.net/cultural/{full_file_name}"
delta_path = "abfss://bronze@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/cultural_places/data"
deltaTable = "dbw_routemind_euskadi_dev.bronze.cultural_places"



df_cult_places_raw = spark.read \
    .option("multiline", "true") \
    .json(source_path)


In [0]:
df_cult_places_bronze = df_cult_places_raw.withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

In [0]:

df_cult_places_bronze.write \
    .format("delta") \
    .option("path", delta_path) \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(deltaTable)


print(f"OVERWRITE completed on {deltaTable}. rows processed: {df_cult_places_bronze.count()}")
